# **EXERCISE 4.1: Naive Bayes**

In [17]:
# Documents

documents = [
    ("SPAM", "Free money now!!!"),
    ("HAM", "Hi mom, how are you?"),
    ("SPAM", "Lowest price for your meds"),
    ("HAM", "Are we still on for dinner?"),
    ("SPAM", "Win a free iPhone today"),
    ("HAM", "Let's catch up tomorrow at the office"),
    ("HAM", "Meeting at 3 PM tomorrow"),
    ("SPAM", "Get 50% off, limited time!"),
    ("HAM", "Team meeting in the office"),
    ("SPAM", "Click here for prizes!"),
    ("HAM", "Can you send the report?")
]

In [18]:
# Remove punctuation and convert to lowercase
import string
def preprocess(text):
    return text.translate(str.maketrans("", "", string.punctuation)).lower()

# Create a set of unique words
vocabulary = set()
for label, text in documents:
    words = preprocess(text).split()
    vocabulary.update(words)
vocabulary = sorted(vocabulary)
print("Vocabulary:", vocabulary)

Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'lets', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']


## **1. Performing it manually. In manually developing a Naïve Bayes model, create methods that will do the following:**

### **1.a. Generate a Bag of Words (for word frequency)**

In [19]:
# Bag of Words for Word Frequency per all documents
from collections import Counter
overall_word_freq = Counter()
for label, text in documents:
    words = preprocess(text).split()
    overall_word_freq.update(words)

print("\n======= Naïve Bayes Word Frequency Analysis =======\n")
print("Overall Word Frequencies:")
print(f"{'Word':<15} {'Frequency':<10}")
for word in sorted(vocabulary):
    print(f"{word:<15} {overall_word_freq[word]:<10}")

# Bag of Words for Word Frequency per class
from collections import defaultdict
word_freq = defaultdict(lambda: {"SPAM": 0, "HAM": 0})
for label, text in documents:
    words = preprocess(text).split()
    for word in words:
        word_freq[word][label] += 1

print("\nWord Frequencies per Class:")
print(f"{'Word':<15} {'SPAM':<10} {'HAM':<10}")
for word in sorted(vocabulary):
    print(f"{word:<15} {word_freq[word]['SPAM']:<10} {word_freq[word]['HAM']:<10}")


======= Naïve Bayes Word Frequency Analysis =======

Overall Word Frequencies:
Word            Frequency 
3               1         
50              1         
a               1         
are             2         
at              2         
can             1         
catch           1         
click           1         
dinner          1         
for             3         
free            2         
get             1         
here            1         
hi              1         
how             1         
in              1         
iphone          1         
lets            1         
limited         1         
lowest          1         
meds            1         
meeting         2         
mom             1         
money           1         
now             1         
off             1         
office          2         
on              1         
pm              1         
price           1         
prizes          1         
report          1         
send            1         
st

### **1.b. Calculate the prior for the class HAM and SPAM**

In [20]:
total_docs = len(documents)
class_counts = Counter(label for label, text in documents)

# Count number of words in each class
ham_word_count = sum(word_freq[word]["HAM"] for word in vocabulary)
spam_word_count = sum(word_freq[word]["SPAM"] for word in vocabulary)

# Count number of documents in each class
ham_docs_count = class_counts["HAM"]
spam_docs_count = class_counts["SPAM"]

prior_ham = ham_docs_count / total_docs
prior_spam = spam_docs_count / total_docs

print("\n======= Prior Probabilities =======")
print(f"Total Documents: {total_docs}")
print(f"Ham Word Count: {ham_word_count}, Spam Word Count: {spam_word_count}")
print(f"Ham Documents: {ham_docs_count}, P(HAM) = {prior_ham:.4f}")
print(f"Spam Documents: {spam_docs_count}, P(SPAM) = {prior_spam:.4f}")


======= Prior Probabilities =======
Total Documents: 11
Ham Word Count: 33, Spam Word Count: 22
Ham Documents: 6, P(HAM) = 0.5455
Spam Documents: 5, P(SPAM) = 0.4545


### **1.c. Calculate the likelihood of the tokens in the vocabulary with respectto the class.**

In [21]:
print("\n======= Likelihoods P(word|class) =======")
print(f"{'Word':<15} {'P(word|HAM)':<15} {'P(word|SPAM)':<15}")
for word in sorted(vocabulary):
    ham_freq = word_freq[word]["HAM"]
    spam_freq = word_freq[word]["SPAM"]
    
    # Laplace smoothing
    likelihood_ham = (ham_freq + 1) / (ham_word_count + len(vocabulary))
    likelihood_spam = (spam_freq + 1) / (spam_word_count + len(vocabulary))
    print(f"{word:<15} {likelihood_ham:<15.4f} {likelihood_spam:<15.4f}")


======= Likelihoods P(word|class) =======
Word            P(word|HAM)     P(word|SPAM)   
3               0.0260          0.0152         
50              0.0130          0.0303         
a               0.0130          0.0303         
are             0.0390          0.0152         
at              0.0390          0.0152         
can             0.0260          0.0152         
catch           0.0260          0.0152         
click           0.0130          0.0303         
dinner          0.0260          0.0152         
for             0.0260          0.0455         
free            0.0130          0.0455         
get             0.0130          0.0303         
here            0.0130          0.0303         
hi              0.0260          0.0152         
how             0.0260          0.0152         
in              0.0260          0.0152         
iphone          0.0130          0.0303         
lets            0.0260          0.0152         
limited         0.0130          0.0303       

### **1.d. Determine the class of the following test sentence:**
- **"Limited offer, click here!"**
- **"Meeting at 2 PM with the manager."**

In [22]:
import math

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]
test_sentences = [preprocess(sentence) for sentence in test_sentences]

print("\n======= Classifying Test Sentences =======\n")
for sentence in test_sentences:
    words = preprocess(sentence).split()
    log_prob_ham = math.log(prior_ham)
    log_prob_spam = math.log(prior_spam)
    
    for word in words:
        if word in vocabulary:
            ham_freq = word_freq[word]["HAM"]
            spam_freq = word_freq[word]["SPAM"]
            likelihood_ham = (ham_freq + 1) / (ham_word_count + len(vocabulary))
            likelihood_spam = (spam_freq + 1) / (spam_word_count + len(vocabulary))
            log_prob_ham += math.log(likelihood_ham)
            log_prob_spam += math.log(likelihood_spam)
    
    predicted_class = "HAM" if log_prob_ham > log_prob_spam else "SPAM"
    print(f"Sentence: '{sentence}'")
    print(f"Predicted Class: {predicted_class}")
    print(f"Log Probability HAM: {log_prob_ham:.4f}, Log Probability SPAM: {log_prob_spam:.4f}\n")


======= Classifying Test Sentences =======

Sentence: 'limited offer click here'
Predicted Class: SPAM
Log Probability HAM: -13.6376, Log Probability SPAM: -11.2780

Sentence: 'meeting at 2 pm with the manager'
Predicted Class: HAM
Log Probability HAM: -13.7047, Log Probability SPAM: -17.5471



## **2.Using Scikit-Learn. Use the scikit-learn package to train and test a Multinomial Naïve Bayes classifer.**

**2.a. Determine the class of the following test sentence:**
- **"Limited offer, click here!"**
- **"Meeting at 2 PM with the manager."**

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# Prepare data for Scikit-Learn
texts = [text for label, text in documents]
labels = [label for label, text in documents]

# Create a pipeline that combines CountVectorizer and MultinomialNB
model = make_pipeline(CountVectorizer(), MultinomialNB())

# Train the model
model.fit(texts, labels)

# Test sentences
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

# Predict classes for test sentences
predicted_classes = model.predict(test_sentences)
probabilities = model.predict_proba(test_sentences)
print("\n======= Scikit-Learn Naïve Bayes Classification =======\n")
for sentence, predicted, prob in zip(test_sentences, predicted_classes, probabilities):
    print(f"Sentence: '{sentence}'")
    print(f"Predicted Class: {predicted}")
    print(f"Probability HAM: {prob[model.classes_ == 'HAM'][0]:.4f}, Probability SPAM: {prob[model.classes_ == 'SPAM'][0]:.4f}\n")


======= Scikit-Learn Naïve Bayes Classification =======

Sentence: 'Limited offer, click here!'
Predicted Class: SPAM
Probability HAM: 0.0847, Probability SPAM: 0.9153

Sentence: 'Meeting at 2 PM with the manager.'
Predicted Class: HAM
Probability HAM: 0.9784, Probability SPAM: 0.0216

